In [1]:
import pickle
import math
from tqdm import tqdm
from collections import Counter, defaultdict
from itertools import combinations

In [2]:
with open("../Data/Output/pipeline_state.pkl", "rb") as f:
    state = pickle.load(f)

In [3]:
MIN_RANKER_CONFIDENCE = 0.90
MIN_NER_CONFIDENCE = 0.5
MIN_OCCURENCES = 3
MIN_LENGTH_OF_WORD = 3

In [4]:
UMLS_FOLDER = "../Data/Ontology/"

MRCONSO_FILE = UMLS_FOLDER + "MRCONSO.RRF"

def prep_UMLS_concepts(MRCONSO_FILE):
  concepts = defaultdict(lambda: {
    "name": None,
    "aliases": set(),
    "name_score": None
  })

  with open(MRCONSO_FILE, encoding="utf-8") as f:
      for line in tqdm(f, desc="Loading MRCONSO"):
          fields = line.rstrip("\n").split("|")

          cui = fields[0]
          lang = fields[1]
          term_type = fields[12]
          term = fields[14]

          if lang != "ENG":
              continue
          # if f"UMLS:{cui}" not in used_cuis:
          #   continue
          if not term.strip():
              continue

          concepts[cui]["aliases"].add(term)

          if (
              concepts[cui]["name"] is None
              and term_type == "PN"
          ):
              concepts[cui]["name"] = term
              
  return concepts

In [5]:
ontology = prep_UMLS_concepts(MRCONSO_FILE)

Loading MRCONSO: 18064970it [00:42, 422256.30it/s]


In [6]:
pairings = []
total_synonyms = 0
add_2nd_alias = 0


for concept in state['linked_annotations'].keys():
    temp_dict = Counter()
    concept_id = state['linked_annotations'][concept][0]['reference_id'][5:]

    for instance in state['linked_annotations'][concept]:
        if (instance['reranker_confidence'] > MIN_RANKER_CONFIDENCE 
            and instance['ner_confidence'] > MIN_NER_CONFIDENCE
            and len(instance['instance_name']) >= MIN_LENGTH_OF_WORD
           ):
            
            synonym_name = instance['instance_name'].lower()
            temp_dict[synonym_name] += 1

    valid_synonyms = [synonym for synonym, occurences in temp_dict.items() if occurences >= MIN_OCCURENCES]
    canonical_name = concept.lower()
    if valid_synonyms and canonical_name not in valid_synonyms:
        valid_synonyms.append(canonical_name)
    if len(valid_synonyms) == 1:
        add_2nd_alias += 1
        for alias in ontology[concept_id]["aliases"]:
            alias = alias.lower()
    
            if alias not in valid_synonyms:
                valid_synonyms.append(alias)
                break

    total_synonyms += len(valid_synonyms)
    for synonym_1, synonym_2 in combinations(valid_synonyms, 2):
        
        pairing = concept_id + "||" + synonym_1 + "||" + synonym_2

        pairings.append(pairing)


In [7]:
print(len(pairings))
print(add_2nd_alias)
print(total_synonyms)
print(len(state['linked_annotations'].keys()))

4302025
29356
742554
919086


In [8]:
with open("../SapBERT_Training/sapbert_train_12.txt", "w", encoding="utf-8") as f:
    for pair in pairings:
        f.write(f"{pair}\n")

In [ ]:
# run the following in terminal
# cd SapBERT_Training/sapbert
# pip install -r requirements.txt
# conda activate sapbert
# cd SapBERT_Training/sapbert/train
# pip uninstall torch torchvision torchaudio -y
pip install torch==1.13.1+cu117 \
 --extra-index-url https://download.pytorch.org/whl/cu117

CUDA_VISIBLE_DEVICES=0 python train.py     --model_dir /mnt/primary/SapBERT_Training/SapBERT_base     --train_dir /mnt/primary/SapBERT_Training/sapbert_train_12.txt     --output_dir /mnt/primary/SapBERT_Training/my_sapbert_12     --use_cuda     --epoch 3     --train_batch_size 256     --learning_rate 2e-5     --max_length 25     --checkpoint_step 999999     --amp     --pairwise     --random_seed 33     --loss ms_loss     --use_miner     --type_of_triplets all     --miner_margin 0.2     --agg_mode cls